# 03 — Design de experimento com power gate
Fluxo: tratadas propostas → exclusão de spillover no donor pool → **análise de
poder por simulação** → só registra como `approved` se MDE ≤ efeito esperado e
pré-registro completo (hipótese + regra de decisão). O `save_experiment` **recusa**
registros que não cumpram o contrato.

In [ ]:
%pip install /Workspace/Shared/supply_experiments/supply_experiments-1.0.2-py3-none-any.whl --quiet --force-reinstall --no-deps
dbutils.library.restartPython()

In [ ]:
from supply_experiments.spark_io import (load_city_panel, TABLES,
                                          active_blocked_cities, ExperimentRecord,
                                          ensure_registry, save_experiment)
from supply_experiments.design.spillover import spillover_exclusions
from supply_experiments.design.treated_selection import (recommend_treated_sets,
                                                         recommendations_frame)
from supply_experiments.design.power import power_analysis
from supply_experiments.estimators.scm import fit_scm
from supply_experiments.estimators.ascm import fit_ascm

POST_DAYS = 35
PRE_DAYS  = 122
ESTIMATOR = fit_ascm             # scm | ascm | sdid

TARGET_ENV = "sandbox"
tables = TABLES[TARGET_ENV]
panel, stats = load_city_panel(spark, "2025-09-01", "2026-06-30")
blocked = active_blocked_cities(spark, tables["registry"])
candidates = [c for c in stats.index if c in panel.cities
              and c not in blocked["treated_active"] | blocked["control_active"]]

## Passo 1 — Recomendação de tratadas (market selection)
O framework ranqueia os conjuntos de cidades **mais mensuráveis** (menor MDE) e
o stakeholder só escolhe uma linha da tabela. Restrições da operação entram como
parâmetros: `states=["SP"]`, `must_include=["GUARULHOS"]`, `exclude=[...]`.

In [ ]:
# ---- RECOMENDAÇÃO DE TRATADAS ------------------------------------------------
# Roda inteiramente com SCM (rápido). O ESTIMATOR pré-registrado (ex.: ascm,
# ~100x mais caro por causa do LOO-CV) entra só no power gate confirmatório.
coords = {}   # opcional: dict city -> (lat, lon) p/ exclusão de spillover por raio
recs = recommend_treated_sets(
    panel, stats, candidates, fit_scm,
    pre_days=PRE_DAYS, post_days=POST_DAYS,
    n_treated=(2, 3),            # tamanhos de conjunto a considerar
    states=None,                 # ex.: ["SP", "MG"] se a operação for regional
    must_include=[],             # cidades que a operação exige tratar
    exclude=[],                  # cidades vetadas pela operação
    coords=coords,
    max_gmv_share_per_city=0.05, # evita tratar cidades grandes demais
)
display(recommendations_frame(recs))

## Passo 2 — Pré-registro
Escolha o `CHOSEN_RANK` da tabela acima. Se as tratadas forem impostas pelo
negócio, defina `TREATED` manualmente e derive `donors` com
`spillover_exclusions(TREATED, candidates, coords, radius_km=40, adjacency=...)`.

In [ ]:
# ---- PRÉ-REGISTRO (preencher ANTES de olhar qualquer resultado) -------------
CHOSEN_RANK      = 1
TREATED          = recs[CHOSEN_RANK - 1].cities
donors           = recs[CHOSEN_RANK - 1].donors
HYPOTHESIS       = "Tratativa de ruptura via chat reduz rupture_rate e protege GMV"
EXPECTED_EFFECT  = 0.05          # efeito esperado no KPI primário (fração)
DECISION_RULE    = "ship se ATT>0 e p<=0.10 (permutação) em >=2 de 3 estimadores"
print("Tratadas:", TREATED, "| doadoras:", len(donors))

In [ ]:
# ---- POWER GATE (confirmação no design final) --------------------------------
pw = power_analysis(panel, TREATED, donors[:20], ESTIMATOR,
                    pre_days=PRE_DAYS, post_days=POST_DAYS,
                    effect_grid=(0.0, 0.02, 0.03, 0.05, 0.08),
                    n_sims_per_point=30, alpha=0.10)
for d, p in sorted(pw.power.items()):
    print(f"  δ={d:5.1%} -> poder {p:6.1%}")
print(f"FPR(δ=0)={pw.fpr:.1%} | MDE@80% = {pw.mde_80}")
assert pw.mde_80 is not None and pw.mde_80 <= EXPECTED_EFFECT, (
    f"SEM PODER: MDE={pw.mde_80} > efeito esperado {EXPECTED_EFFECT}. "
    "Aumente cidades tratadas, duração, ou escolha cidades menos ruidosas.")

In [ ]:
# ---- registro (validado) -----------------------------------------------------
import uuid
from datetime import date, timedelta

start = date.today() + timedelta(days=7)   # início planejado
rec = ExperimentRecord(
    experiment_id=str(uuid.uuid4()), name="tratativa_chat_ruptura",
    hypothesis=HYPOTHESIS, status="approved", design_method="ascm",
    primary_kpi="gmv", guardrail_kpis=["rupture_rate_order"],
    start_date=start, end_date=start + timedelta(days=POST_DAYS - 1),
    pre_window_days=PRE_DAYS,
    treated_units=[{"city_norm": c, "city_address": c, "state_address": "SP",
                    "weight": None} for c in TREATED],
    control_units=[{"city_norm": c, "city_address": c,
                    "state_address": str(stats.loc[c, "state_address"]),
                    "weight": None} for c in donors[:20]],
    mde_estimated=pw.mde_80, expected_effect=EXPECTED_EFFECT,
    decision_rule=DECISION_RULE,
    created_by=spark.sql("SELECT current_user()").first()[0],
)
ensure_registry(spark, tables["registry"])
save_experiment(spark, tables["registry"], rec)   # levanta erro se pré-registro incompleto
print("Experimento registrado:", rec.experiment_id)